# Reading the signals behind Bangladesh's measles resurgence

**Question:** How unusual was the reported increase, and how do case definitions
and vaccination-coverage denominators affect interpretation?

This notebook explains the public health case study in a reproducible sequence. It
uses a frozen August 2026 WHO-derived extract published by Pranta Sikder (Zenodo
22167512, v1.1.0). Monthly case reports cover all ages; vaccination indicators refer
to their own child target populations.

This is exploratory reanalysis after data inspection. It does not identify a cause
of the epidemic or estimate real-time warning performance. No model/API calls run here.


In [1]:
from pathlib import Path
import json
import pandas as pd
from measleswatch.describe import read_validated_tables
from measleswatch.analysis import matched_periods, magnitude, coverage_accounting, case_composition, independent_raw_checks
from measleswatch.study_figures import load_run

root = Path.cwd()
if not (root / ".devin/descriptive_analysis.toml").exists():
    root = root.parent
assert (root / ".devin/descriptive_analysis.toml").is_file(), "Run from the repository or notebooks directory"
tables, source_manifest = read_validated_tables(root / "data/processed/phase1-r2")
saved = load_run(root)
cases, coverage = tables["surveillance_monthly"], tables["coverage_annual"]
print(f"Validated {len(cases)} monthly category records and {len(coverage)} source-specific coverage records.")


Validated 875 monthly category records and 81 source-specific coverage records.


## 1. Make the historical comparison fair

Comparing five months in 2026 with a complete historical year would mix observation
windows. Choosing only the low 2025 total would also give an incomplete historical
perspective. We retain the same January–May window in all 14 previous years.


In [2]:
periods = matched_periods(cases, "measles_lab_confirmed", end_month=5)
comparison = magnitude(periods)
print(pd.DataFrame(periods)[["year", "count", "reporting_months"]].to_string(index=False))
print(f"2026: {comparison['current_count']:,} laboratory-confirmed reports")
print(f"Previous maximum: {comparison['previous_maximum']:,}, in {comparison['previous_maximum_years']}")
print(f"Ratio: {comparison['ratio_to_previous_maximum']:.2f} times the previous matched-period maximum")


 year  count  reporting_months
 2012    497                 5
 2013     33                 5
 2014     26                 5
 2015     77                 5
 2016    226                 5
 2017   1817                 5
 2018    958                 5
 2019   1345                 5
 2020   1245                 5
 2021     29                 5
 2022     60                 5
 2023    110                 5
 2024     94                 5
 2025     39                 5
 2026  11905                 5
2026: 11,905 laboratory-confirmed reports
Previous maximum: 1,817, in [2017]
Ratio: 6.55 times the previous matched-period maximum


![Calendar-aligned historical comparison](../results/study-figures/01_resurgence.png)

The 2026 rise remains large against the highest previous matched period. Gray curves
are observed historical years, not confidence or prediction intervals. Counts still
depend on investigation, testing and reporting; all 2026 reports are provisional.

## 2. Check whether the endpoint or case definition changes the conclusion

Retain every declared March/April/May comparison under both outcomes. A result is
robust only to the choices actually checked; this does not eliminate surveillance bias.


In [3]:
checks = []
all_periods = []
for measure in ("measles_lab_confirmed", "measles_total_classified"):
    for last_month in (3, 4, 5):
        selected = matched_periods(cases, measure, end_month=last_month)
        all_periods.extend(selected)
        result = magnitude(selected)
        checks.append({"Through month": last_month, "Outcome": measure,
                       "2026 reports": result["current_count"],
                       "Previous maximum": result["previous_maximum"],
                       "Ratio": round(result["ratio_to_previous_maximum"], 2)})
print(pd.DataFrame(checks).to_string(index=False))


 Through month                  Outcome  2026 reports  Previous maximum  Ratio
             3    measles_lab_confirmed          1947              1219   1.60
             4    measles_lab_confirmed          7056              1482   4.76
             5    measles_lab_confirmed         11905              1817   6.55
             3 measles_total_classified          5924              2162   2.74
             4 measles_total_classified         29613              2294  12.91
             5 measles_total_classified         50040              2885  17.34


## 3. Read coverage sources separately

Administrative coverage is reported routine doses divided by the reported target
population. The national official estimate and the WHO/UNICEF estimate are different
measures. They may use related evidence and are not three independent trials.


In [4]:
cov = pd.DataFrame(coverage)
recent = cov[cov.year.isin([2024, 2025])]
print(recent.pivot(index=["antigen", "year"], columns="coverage_source", values="reported_pct").to_string())


coverage_source    admin  official  wuenic
antigen year                              
MCV1    2024     123.510      86.1    96.0
        2025      92.658      86.1    90.0
MCV2    2024     121.376      80.7    93.0
        2025      90.695      80.7    86.0


![Separate coverage measures](../results/study-figures/02_coverage.png)

Administrative values above 100% are retained, not clipped. Missing years remain
gaps, not zero coverage. Source disagreement is not a confidence interval.

## 4. Open up the coverage calculation

The source author already documented this accounting issue. We reproduce it from
the input values and reconcile the key records with WHO. A larger target reduces
the percentage even if the dose count is unchanged; fewer doses reduce it as well.


In [5]:
index = {(r["year"], r["antigen"], r["coverage_source"]): r for r in coverage}
accounting = [coverage_accounting(index[(2024, a, "admin")], index[(2025, a, "admin")])
              for a in ("MCV1", "MCV2")]
for item in accounting:
    print(item["antigen"])
    print(f"  Doses: {item['old_doses']:,} to {item['new_doses']:,}; {item['doses_change_percent']:+.2f}%")
    print(f"  Target: {item['old_target']:,} to {item['new_target']:,}; {item['target_change_percent']:+.2f}%")
    print(f"  Target share of log change: {100 * item['target_share_of_log_change']:.1f}%")
    print(f"  Accounting identity residual: {item['identity_residual']:.2e}")


MCV1
  Doses: 4,065,933 to 3,791,393; -6.75%
  Target: 3,292,000 to 4,091,813; +24.30%
  Target share of log change: 75.7%
  Accounting identity residual: 0.00e+00
MCV2
  Doses: 3,995,682 to 3,711,076; -7.12%
  Target: 3,292,000 to 4,091,813; +24.30%
  Target share of log change: 74.6%
  Accounting identity residual: 0.00e+00


![Dose and target changes](../results/study-figures/03_denominator.png)

The decomposition uses `log(C1/C0) = log(D1/D0) - log(T1/T0)`. Its shares describe
logarithmic change in a ratio, not causes of the outbreak or unique children missing
vaccination. WHO's target descriptions differ between the years; that warrants a
comparability check, not an unsupported claim that the new estimate is wrong.

## 5. Keep case classifications separate

The classified total is clinical + epidemiologically linked + laboratory-confirmed.
Suspected reports are not added to that total. The laboratory share is a composition
measure; we lack a testing denominator needed for positivity.


In [6]:
composition = []
for year in (2025, 2026):
    composition.extend(case_composition(cases, f"{year}-01", f"{year}-05"))
print(pd.DataFrame(composition)[["period_start", "measure", "count", "classified_total", "share_percent"]].to_string(index=False))


period_start               measure  count  classified_total  share_percent
     2025-01      measles_clinical      0                39       0.000000
     2025-01    measles_epi_linked      0                39       0.000000
     2025-01 measles_lab_confirmed     39                39     100.000000
     2026-01      measles_clinical   7073             50040      14.134692
     2026-01    measles_epi_linked  31062             50040      62.074341
     2026-01 measles_lab_confirmed  11905             50040      23.790967


![Classification composition](../results/study-figures/04_classification.png)

## 6. Verify arithmetic through a separate path

The following check uses the original CSV files and Decimal arithmetic, rather than
aggregating the canonical JSON again. The current WHO source check is saved separately;
it never replaces the historical August extract or runs automatically here.


In [7]:
raw_checks = independent_raw_checks(root, all_periods, accounting)
assert all(r["agrees"] for r in raw_checks)
source_check = json.loads((root / "sources/context/wiise-2026-10-27/manifest.json").read_text())
assert source_check["status"] == "reconciled"
print(f"Independent raw-source checks passed: {len(raw_checks)}")
print(f"WHO coverage-value comparisons reconciled: {source_check['comparisons']}")


Independent raw-source checks passed: 98
WHO coverage-value comparisons reconciled: 20


## What we learned, and what to investigate next

The reported rise remains large under matched calendar periods and both checked case
definitions. Coverage interpretation improves when dose volumes and target changes
are examined together. This is useful evidence synthesis, with the source author's
denominator analysis explicitly credited.

Age/district breakdowns, testing volumes, investigation records, target documentation
and historical release vintages would help explain mechanisms. The present aggregate
data do not identify individual immunity, causation, or prospective warning skill.

Read the [research brief](../reports/public_health_brief.md),
[methods note](../reports/methods.md), and [source attribution](../DATA_SOURCES.md).
